In [1]:
# Solution of a multi-product supply chain (SC) problem in Julia
# We now let the model decide where to install technologies
using JuMP, HiGHS

# sets 
P = ["DM","Compost"]
S = ["DF"] 
D = ["CF", "SF","DC"]
L = ["DF->CF","DF->SF","DF->Composter","Composter->DC"]
T = ["Composter"]

# supply data
sprod = ["DM"]
sprod = Dict(zip(S,sprod))
sub = [1000] # ton
sub = Dict(zip(S,sub))
sbid = [-0.7] # $/ton
sbid = Dict(zip(S,sbid))

#demand data
dprod = ["DM","DM","Compost"]
dprod = Dict(zip(D,dprod))
dub = [500,500,100] # ton 
dub = Dict(zip(D,dub))
dbid = [-0.5,1.5,100] # $/ton
dbid = Dict(zip(D,dbid))

# transport data
flocs = ["DF","DF","DF","Composter"]
flocr = ["CF","SF","Composter","DC"]
flocs = Dict(zip(L,flocs))
flocr = Dict(zip(L,flocr))
fub = [1000,1000,1000,1000] # ton
fub = Dict(zip(L,fub))
fprod = ["DM","DM","DM","Compost"]
fprod = Dict(zip(L,fprod))
fbid = [0.1,0.2,0.0,1.0] # $/ton
fbid = Dict(zip(L,fbid))

# technology data
T = ["Composter"]

ξub = [500]; # tons
ξub = Dict(zip(T,ξub))

ξbid = [1]; # $/ton
ξbid = Dict(zip(T,ξbid))

# reference product for Composter is Manure
# ["DM","Compost"]
γ = [-1.0 0.1]
γ = Dict((T[i], P[j]) => γ[i, j] for j in eachindex(P), i in eachindex(T))

Dict{Tuple{String, String}, Float64} with 2 entries:
  ("Composter", "Compost") => 0.1
  ("Composter", "DM")      => -1.0

In [2]:
P,S,D,T

(["DM", "Compost"], ["DF"], ["CF", "SF", "DC"], ["Composter"])

In [3]:
# solve model with fixed installation decisions
m = Model(HiGHS.Optimizer); 

# variables
@variable(m, s[S]>=0)
@variable(m, d[D]>=0)
@variable(m, f[L]>=0)
@variable(m, ξ[T]>=0)

# capacity constraints
@constraint(m, [i in S], s[i] <= sub[i])
@constraint(m, [j in D], d[j] <= dub[j])
@constraint(m, [l in L], f[l] <= fub[l])
@constraint(m, [t in T], ξ[t] <= ξub[t]) 

# supply balance constraints
@constraint(m, sbal[i in S],  s[i]  ==  sum(f[l] for l in L if flocs[l] == i && fprod[l] == sprod[i]))

# demand balance constraints
@constraint(m, dbal[j in D],  d[j]  ==  sum(f[l] for l in L if flocr[l] == j && fprod[l] == dprod[j]))

# tech balance constraints
@constraint(m, 
    bal[t in T,p in P],     sum(f[l] for l in L if flocr[l] == t && fprod[l] == p) 
                         +  ξ[t]*γ[t,p] 
                        ==  sum(f[l] for l in L if flocs[l] == t && fprod[l] == p))

#objective function  
dcost = @expression(m, sum(dbid[j]*d[j] for j in D))
scost = @expression(m, sum(sbid[i]*s[i] for i in S))
fcost = @expression(m, sum(fbid[l]*f[l] for l in L))
ξcost = @expression(m, sum(ξbid[t]*ξ[t] for t in T))

@objective(m, Max, dcost - scost - fcost - ξcost)

print(m)

In [4]:
optimize!(m)


Presolving model
1 rows, 4 cols, 4 nonzeros
1 rows, 4 cols, 4 nonzeros
Presolve : Reductions: rows 1(-14); columns 4(-5); elements 4(-19)
Solving the presolved LP
Using EKK dual simplex solver - serial
LP  has all |entries|=1; max column count = 1 (limit 24); average column count = 1 (limit 6): So is a candidate for LiDSE
  Iteration        Objective     Infeasibilities num(sum)
          0    -5.8000406941e+03 Pr: 0(0) 0s
          0    -5.8000000000e+03 Pr: 0(0) 0s
Solving the original LP from the solution after postsolve
Model   status      : Optimal
Objective value     :  5.8000000000e+03
HiGHS run time      :          0.00


In [5]:
# get results for all suppliers

# allocations
sa = zeros(length(S))
sa = Dict(zip(S,sa))
for i in S
    sa[i]=value.(s)[i]
    print("sa[",i,"] = ",sa[i]," unit \n")
end

sa[DF] = 1000.0 unit 


In [6]:
# get results for all consumers

da = zeros(length(D))
da = Dict(zip(D,da))
for i in D
    da[i]=value.(d)[i]    
    print("da[",i,"] = ",da[i]," unit \n")
end


da[CF] = 0.0 unit 
da[SF] = 500.0 unit 
da[DC] = 50.0 unit 


In [7]:
# get results for all transporters

fa = zeros(length(L))
fa = Dict(zip(L,fa))
for l in L
    fa[l]=value.(f)[l]    
    print("fa[",l,"] = ",fa[l]," unit \n")
end

fa[DF->CF] = -0.0 unit 
fa[DF->SF] = 500.0 unit 
fa[DF->Composter] = 500.0 unit 
fa[Composter->DC] = 50.0 unit 
